# درس 13: بهینه‌سازی، کمترین مربعات و Kelly

متن کامل درس: [https://quantacademy.ir/courses/quant-math/optimization-least-squares-kelly/](https://quantacademy.ir/courses/quant-math/optimization-least-squares-kelly/)

داده‌های این دوره **شبیه‌سازی‌شده** هستند، نه قیمت واقعی بازار؛ هدف یاد دادن روش است. آموزشی است و توصیه سرمایه‌گذاری نیست.

سلول‌ها را از بالا به پایین اجرا کن (Shift+Enter).

## ۱. مشتق و گرادیان

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

def f(x):
    return (x - 3) ** 2 + 1

def grad(x):
    return 2 * (x - 3)

for lr in (0.01, 0.1, 1.1):
    x = 10.0
    for _ in range(30):
        x = x - lr * grad(x)
    print(lr, round(x, 4), round(f(x), 4))

## ۲. کمترین مربعات، یک بهینه‌سازی است

In [ ]:
rng = np.random.default_rng(13)
n = 500
xs = rng.normal(0, 1, n)
y = 2.0 + 0.5 * xs + rng.normal(0, 1, n)
X = np.column_stack([np.ones(n), xs])

beta_ne = np.linalg.solve(X.T @ X, X.T @ y)
print(beta_ne.round(4))

b = np.zeros(2)
for _ in range(500):
    g = 2 / n * X.T @ (X @ b - y)
    b = b - 0.1 * g
print(b.round(4))
print(np.linalg.lstsq(X, y, rcond=None)[0].round(4))

## ۳. بهینه‌سازی با قید: پرتفوی کمینه‌واریانس

In [ ]:
vol = np.array([0.20, 0.30, 0.40])
corr = np.array([[1, 0.9, 0.5], [0.9, 1, 0.6], [0.5, 0.6, 1]])
Sigma = np.outer(vol, vol) * corr
ones = np.ones(3)
inv = np.linalg.inv(Sigma)

w_cf = inv @ ones / (ones @ inv @ ones)
print(w_cf.round(4), round(np.sqrt(w_cf @ Sigma @ w_cf), 4), round(np.sqrt(1 / (ones @ inv @ ones)), 4))

cons = [{"type": "eq", "fun": lambda w: w.sum() - 1}]
res = minimize(lambda w: w @ Sigma @ w, np.full(3, 1 / 3), method="SLSQP",
               constraints=cons, options={"ftol": 1e-14})
print(res.x.round(4), round(np.sqrt(res.fun), 4))

lo = minimize(lambda w: w @ Sigma @ w, np.full(3, 1 / 3), method="SLSQP",
              bounds=[(0, 1)] * 3, constraints=cons, options={"ftol": 1e-14})
print(lo.x.round(4), round(np.sqrt(lo.fun), 4))

w_eq = np.full(3, 1 / 3)
print(round(np.sqrt(w_eq @ Sigma @ w_eq), 4))

## ۴. خطای تخمین، دشمن بهینه‌ساز

In [ ]:
def true_cov(n_assets):
    r = np.random.default_rng(1)
    B = r.normal(0.8, 0.3, n_assets)
    return 0.02 ** 2 * np.outer(B, B) + np.diag(np.full(n_assets, 0.015 ** 2))

n_assets = 20
S_true = true_cov(n_assets)
L = np.linalg.cholesky(S_true)

def minvar(S):
    w = np.linalg.solve(S, np.ones(len(S)))
    return w / w.sum()

def risk(w):
    return np.sqrt(w @ S_true @ w)

ideal = risk(minvar(S_true))
rng = np.random.default_rng(4)
for T in (40, 100, 250, 1000):
    out = []
    for _ in range(300):
        Z = rng.normal(size=(T, n_assets)) @ L.T
        out.append(risk(minvar(np.cov(Z, rowvar=False))))
    print(T, round(np.median(out) / ideal, 3))
print("equal weight", round(risk(np.full(n_assets, 1 / n_assets)) / ideal, 3))

## ۵. معیار Kelly: چه کسری از سرمایه؟

In [ ]:
p, b_odds = 0.55, 1.0
f_star = p - (1 - p) / b_odds

def growth(f):
    return p * np.log(1 + b_odds * f) + (1 - p) * np.log(1 - f)

print(round(f_star, 4))
for f in (0.0, 0.05, 0.1, 0.15, 0.2, 0.3):
    print(f, round(growth(f), 5))

In [ ]:
rng = np.random.default_rng(5)
paths, T = 5000, 1000
wins = rng.random((paths, T)) < p
print("fraction  f      median_wealth  P(end<1)  median_max_drawdown")
for label, frac in (("quarter", 0.25), ("half", 0.5), ("full", 1.0), ("double", 2.0)):
    f = frac * f_star
    logs = np.where(wins, np.log(1 + b_odds * f), np.log(1 - f))
    wealth_path = np.exp(np.cumsum(logs, axis=1))
    peak = np.maximum.accumulate(np.maximum(wealth_path, 1.0), axis=1)
    mdd = (wealth_path / peak - 1).min(axis=1)
    final = wealth_path[:, -1]
    print(f"{label:<9} {f:<6.3f} {np.median(final):<14.2f} {(final < 1).mean():<9.3f} {-np.median(mdd):.3f}")

## ۶. Kelly برای بازده پیوسته و هشدار برآورد

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/quantacademyir/quant-math/main/data/btc_daily_5y.csv", parse_dates=["date"], index_col="date")
ret = df["close"].pct_change().dropna()
print(round(ret.mean(), 5), round(ret.var(), 5), round(ret.mean() / ret.var(), 2))

by_year = ret.groupby(ret.index.year).agg(lambda r: r.mean() / r.var())
print(by_year.round(2).to_dict())
print(round(ret.std() / np.sqrt(365), 5))